# Free GPU video-generation smoke test: WanGP + Wan 2.1 1.3B

Goal: generate one **real ~5-second 480p clip** on a free Kaggle GPU, with no paid API.

Recommended Kaggle settings before running: **Accelerator → GPU** and **Internet → On**.

The notebook uses WanGP's low-VRAM/older-GPU fallback: SDPA attention, profile 4, FP16.

In [ ]:
import os, subprocess, sys, json, time, textwrap, pathlib
print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=False)
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('capability:', torch.cuda.get_device_capability(0))
        print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
except Exception as e:
    print('torch probe error:', e)


## Install WanGP
We keep Kaggle's CUDA driver and use WanGP's compatibility path rather than Flash Attention.

In [ ]:
%%bash
set -euxo pipefail
cd /kaggle/working
rm -rf Wan2GP
git clone --depth 1 https://github.com/deepbeepmeep/Wan2GP.git
cd Wan2GP
python -m pip install -U pip setuptools wheel
# WanGP's requirements; keep existing Kaggle torch unless dependency resolution requires otherwise.
python -m pip install -r requirements.txt


## Generate one production-shaped test clip
The script discovers the installed Wan 1.3B text-to-video model dynamically, starts from its own defaults, then requests **81 frames** at **480p**. Wan-family models normally use 16 fps, so 81 frames is about 5 seconds.

In [ ]:
from pathlib import Path
import json, time, sys, os

ROOT = Path("/kaggle/working/Wan2GP")
OUT = Path("/kaggle/working/wan_free_gpu_outputs")
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from shared.api import init

started = time.time()
session = init(
    root=ROOT,
    output_dir=OUT,
    cli_args=["--t2v-1-3B", "--attention", "sdpa", "--profile", "4", "--fp16"],
    console_output=True,
)

models = session.list_model_metadata(main_output="video")
candidates = []
for m in models:
    text = json.dumps(m, ensure_ascii=False).lower()
    # Prefer Wan-family 1.3B text-to-video variants.
    score = 0
    if "wan" in text: score += 3
    if "1.3" in text or "1_3" in text or "1-3" in text: score += 5
    if "t2v" in text or "text_to_video" in text: score += 4
    if score:
        candidates.append((score, m))

if not candidates:
    raise RuntimeError("No Wan 1.3B-like video model found. First models: " + json.dumps(models[:5], indent=2))

candidates.sort(key=lambda x: x[0], reverse=True)
model = candidates[0][1]
model_type = model["model_type"]
print("Selected model:", json.dumps(model, indent=2))

settings = session.get_default_settings(model_type)
settings.update({
    "model_type": model_type,
    "prompt": (
        "A single young child in a red shirt is visible from the first frame in a sunny garden. "
        "The same child walks naturally three clear steps toward a small wilted plant, remains visible "
        "throughout, then stops beside the plant. Static camera, full body, continuous motion, no cuts, "
        "no teleporting, no extra people, consistent face and clothing."
    ),
    "resolution": "832x480",
    "video_length": 81,
    "num_inference_steps": 20,
})

print("Effective settings:", json.dumps(settings, indent=2, default=str))
with open("/kaggle/working/wan_test_settings.json", "w") as f:
    json.dump(settings, f, indent=2, default=str)

job = session.submit_task(settings)
for event in job.events.iter(timeout=0.2):
    if event.kind == "progress":
        p = event.data
        print("PROGRESS", getattr(p, "phase", None), getattr(p, "progress", None),
              getattr(p, "current_step", None), "/", getattr(p, "total_steps", None))
    elif event.kind == "status":
        print("STATUS", event.data)
    elif event.kind == "stream":
        line = event.data
        print(f"[{line.stream}] {line.text}")

result = job.result()
elapsed = time.time() - started
summary = {
    "success": bool(result.success),
    "elapsed_seconds": elapsed,
    "generated_files": [str(x) for x in result.generated_files],
    "errors": [getattr(e, "message", str(e)) for e in result.errors],
    "selected_model": model,
}
print(json.dumps(summary, indent=2, default=str))
with open("/kaggle/working/wan_test_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

if not result.success:
    raise RuntimeError("WanGP generation failed: " + repr(summary["errors"]))


## Inspect the result
This verifies duration, resolution, FPS, and writes a contact sheet for quick continuity inspection.

In [ ]:
import glob, json, os, subprocess, pathlib
files = sorted(glob.glob("/kaggle/working/wan_free_gpu_outputs/**/*", recursive=True))
videos = [p for p in files if p.lower().endswith((".mp4",".webm",".mov"))]
print("Video files:", videos)
if not videos:
    raise RuntimeError("No video output found")
video = videos[-1]
subprocess.run([
    "ffprobe","-v","error",
    "-show_entries","stream=codec_name,width,height,r_frame_rate,nb_frames:format=duration,size,bit_rate",
    "-of","json",video
], check=False)
print("OUTPUT_VIDEO="+video)
